In [ ]:
load_date = "2014-05-02"

In [ ]:
from pyspark.sql.functions import col, date_sub, lit
from pyspark.sql.types import DateType


def read_customer_snapshot(path, snapshot_date):
    return (
        spark.read.parquet(path)
        .select(
            col("CustomerID").cast("long").alias("customer_id"),
            col("CustomerName").alias("customer_name"),
            col("CustomerType").alias("customer_type"),
            col("TerritoryID").cast("long").alias("territory_id"),
        )
        .withColumn(
            "change_date",
            lit(snapshot_date).cast(DateType()),
        )
    )


def apply_scd2(current_history, changes):
    current = current_history.filter(col("is_current"))

    comparison = (
        current.alias("old")
        .join(
            changes.alias("new"),
            col("old.customer_id") == col("new.customer_id"),
            "full",
        )
    )

    different_values = ~(
        col("old.customer_name").eqNullSafe(col("new.customer_name"))
        & col("old.customer_type").eqNullSafe(col("new.customer_type"))
        & col("old.territory_id").eqNullSafe(col("new.territory_id"))
    )

    changed_or_new = comparison.filter(
        col("new.customer_id").isNotNull()
        & (
            col("old.customer_id").isNull()
            | different_values
        )
    )

    changed_existing = changed_or_new.filter(
        col("old.customer_id").isNotNull()
    )

    changed_ids = changed_or_new.select(
        col("new.customer_id").alias("customer_id")
    ).distinct()

    historical = current_history.filter(~col("is_current"))

    unchanged_current = current.join(
        changed_ids,
        "customer_id",
        "left_anti",
    )

    unchanged_versions = historical.unionByName(
        unchanged_current
    )

    closed_versions = changed_existing.select(
        col("old.customer_id").alias("customer_id"),
        col("old.customer_name").alias("customer_name"),
        col("old.customer_type").alias("customer_type"),
        col("old.territory_id").alias("territory_id"),
        col("old.valid_from").alias("valid_from"),
        date_sub(
            col("new.change_date"),
            1,
        ).alias("valid_to"),
        lit(False).alias("is_current"),
        col("old.source_system").alias("source_system"),
    )

    new_versions = changed_or_new.select(
        col("new.customer_id").alias("customer_id"),
        col("new.customer_name").alias("customer_name"),
        col("new.customer_type").alias("customer_type"),
        col("new.territory_id").alias("territory_id"),
        col("new.change_date").alias("valid_from"),
        lit("9999-12-31").cast(DateType()).alias("valid_to"),
        lit(True).alias("is_current"),
        lit("adls_landing").alias("source_system"),
    )

    return (
        unchanged_versions
        .unionByName(closed_versions)
        .unionByName(new_versions)
    )


j0 = (
    spark.read.parquet(
        "Files/landing/"
        "Customer/load_date=2014-04-30/customer_j0.parquet"
    )
    .select(
        col("CustomerID").cast("long").alias("customer_id"),
        col("CustomerName").alias("customer_name"),
        col("CustomerType").alias("customer_type"),
        col("TerritoryID").cast("long").alias("territory_id"),
    )
    .withColumn(
        "valid_from",
        lit("2011-05-31").cast(DateType()),
    )
    .withColumn(
        "valid_to",
        lit("9999-12-31").cast(DateType()),
    )
    .withColumn("is_current", lit(True))
    .withColumn("source_system", lit("adls_landing"))
)

customer_history = j0

changes_path = (
    "Files/landing/"
    f"Customer/load_date={load_date}/Customer.parquet"
)

changes = read_customer_snapshot(changes_path, load_date)

customer_history = spark.read.table("silver_customer_scd2")

customer_history = apply_scd2(
    customer_history,
    changes,
)

(
    customer_history.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver_customer_scd2")
)

print(f"Date appliquée : {load_date}")
print(f"Lignes SCD2 : {customer_history.count()}")

print(
    "Versions courantes : "
    f"{customer_history.filter(col('is_current')).count()}"
)
print("Table créée : silver_customer_scd2")

In [ ]:
duplicate_current = (
    customer_history
    .filter(col("is_current"))
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

invalid_periods = customer_history.filter(
    col("valid_to") < col("valid_from")
).count()

print(f"Clients avec plusieurs versions courantes : {duplicate_current}")
print(f"Périodes invalides : {invalid_periods}")

if duplicate_current > 0:
    raise ValueError("Plusieurs versions courantes pour un même client.")

if invalid_periods > 0:
    raise ValueError("Des périodes SCD2 sont invalides.")

print("Contrôles SCD2 Fabric réussis.")